# Collect GVGAI ASCII shards on Colab

Runs the Java MCTS collector (`tracks.singlePlayer.ascii.RunAsciiCollectionMCTS`) on Colab's CPU. Shards are written to the VM's **local SSD** at wire speed; a background uploader (`data.upload_shards`) rsyncs each `shard_XXXXX/` to Google Drive the moment the Java writer finalizes it, under the canonical layout shared with `train_ascii_vae_colab.ipynb`:

```
MyDrive/DecoupliWo/data/transitions/{train,test}/<game>/<variant>/shard_XXXXX/
```

Why not write straight to the Drive FUSE mount? Concurrent `.npy` writes from multiple Java processes via Drive FUSE are slow and unreliable (stalls, partial files). Writing locally + per-shard rsync is fast and resumable.

Game ordering is round-robin: the collector interleaves `(game, variant, split)` jobs across games before launching, so shards from every game begin appearing on Drive roughly in parallel — the training notebook will therefore see a mix of games in every mini-batch even while collection is still running.

## Runtime

- `Runtime` → `Change runtime type` → **CPU** (GPU is wasted here; MCTS is single-threaded CPU-bound)
- More vCPUs = faster collection. Colab High-RAM / Pro runtimes give more cores which `--jobs` scales across.

## One-time prerequisite on Drive

Your local `gvgai/` is an embedded git repo inside DecoupliWo (not a tracked submodule), so `git clone DecoupliWo` on Colab does NOT bring the GVGAI source. You must upload a tarball of your local `gvgai/` directory to Drive **once**. On your Mac:

```bash
cd ~/Documents/GitHub/DecoupliWo
tar -czf gvgai.tar.gz \
  gvgai/src \
  gvgai/examples \
  gvgai/sprites \
  gvgai/out \
  gvgai/gson-2.6.2.jar
```

Verify it contains the compiled collector class:

```bash
tar -tzf gvgai.tar.gz | grep 'RunAsciiCollectionMCTS.class'
# expected: gvgai/out/production/gvgai/tracks/singlePlayer/ascii/RunAsciiCollectionMCTS.class
```

Upload `gvgai.tar.gz` to `MyDrive/DecoupliWo/gvgai.tar.gz`. Cell 4 asserts it exists; Cell 8 extracts it into `/content/DecoupliWo/gvgai/` on each session.

The long-term fix (out of scope here) is to either make `gvgai/` a proper git submodule or push it to its own GitHub repo so it clones automatically.

## 1. Mount Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Canonical paths (shared with the training notebook)

In [ ]:
import os
from pathlib import Path

REPO_URL = 'https://github.com/wessimpson/DecoupliWo.git'
REPO_BRANCH = 'gameNGen-world_model-v2'
REPO_DIR = Path('/content/DecoupliWo')

DRIVE_ROOT = Path('/content/drive/MyDrive/DecoupliWo')
DRIVE_GVGAI_TAR = DRIVE_ROOT / 'gvgai.tar.gz'
DRIVE_DATA_DIR = DRIVE_ROOT / 'data' / 'transitions'

LOCAL_DATA_DIR = REPO_DIR / 'data' / 'transitions'
UPLOADER_STOP_FILE = Path('/tmp/upload_shards.stop')
UPLOADER_LOG = Path('/tmp/upload_shards.log')

assert DRIVE_ROOT.parent.is_dir(), (
    f'Expected Drive to be mounted at {DRIVE_ROOT.parent}. '
    f'Re-run the "Mount Drive" cell above.'
)
DRIVE_DATA_DIR.mkdir(parents=True, exist_ok=True)
(DRIVE_DATA_DIR / 'train').mkdir(parents=True, exist_ok=True)
(DRIVE_DATA_DIR / 'test').mkdir(parents=True, exist_ok=True)

assert DRIVE_GVGAI_TAR.is_file(), (
    f'Expected {DRIVE_GVGAI_TAR} on Drive. See the notebook header for the '
    f'one-time tar command on your laptop.'
)
print('OK gvgai bundle  :', DRIVE_GVGAI_TAR, f'({DRIVE_GVGAI_TAR.stat().st_size / 1e6:.1f} MB)')
print('OK drive data dir:', DRIVE_DATA_DIR)
print('planned local dir:', LOCAL_DATA_DIR, '(created after clone)')

## 3. Clone the repo

In [ ]:
import shutil
import subprocess

def _is_git_checkout(path: Path) -> bool:
    return (path / '.git').is_dir()

if REPO_DIR.is_dir() and not _is_git_checkout(REPO_DIR):
    print(f'removing non-git directory at {REPO_DIR}')
    shutil.rmtree(REPO_DIR)

if REPO_DIR.is_dir():
    subprocess.run(['git', '-C', str(REPO_DIR), 'fetch', '--all'], check=True)
    subprocess.run(['git', '-C', str(REPO_DIR), 'checkout', REPO_BRANCH], check=True)
    subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only'], check=True)
else:
    subprocess.run(
        ['git', 'clone', '--branch', REPO_BRANCH, '--single-branch', REPO_URL, str(REPO_DIR)],
        check=True,
    )

os.chdir(REPO_DIR)
print('cwd =', os.getcwd())
subprocess.run(['git', 'log', '-1', '--oneline'], check=True)

## 4. Extract the prebuilt GVGAI bundle into the repo

Extracts `MyDrive/DecoupliWo/gvgai.tar.gz` into `/content/DecoupliWo/gvgai/`. The tar includes the already-compiled `out/production/gvgai/**/*.class` so Colab doesn't need `javac`. Idempotent: if the collector class already exists (e.g. you re-ran the cell), it skips.

In [ ]:
import tarfile

GVGAI_DIR = REPO_DIR / 'gvgai'
COLLECTOR_CLASS = GVGAI_DIR / 'out' / 'production' / 'gvgai' / 'tracks' / 'singlePlayer' / 'ascii' / 'RunAsciiCollectionMCTS.class'

if COLLECTOR_CLASS.is_file():
    print(f'already extracted -> {GVGAI_DIR}')
else:
    with tarfile.open(DRIVE_GVGAI_TAR, 'r:gz') as tf:
        tf.extractall(REPO_DIR)
    print(f'extracted -> {GVGAI_DIR}')

assert COLLECTOR_CLASS.is_file(), (
    f'Expected compiled class at {COLLECTOR_CLASS} after extraction. '
    f'Rebuild the tar on your laptop so it includes gvgai/out/production/gvgai/**.'
)
print('OK compiled class:', COLLECTOR_CLASS)

## 5. Verify Java is available

In [ ]:
!java -version
!javac -version || true

If the cell above failed (no JRE on this VM), uncomment the install line in the next cell and re-run. Skip if it already worked.

In [ ]:
# !apt-get update -qq && apt-get install -y -qq default-jre

## 6. Collect + upload

Two processes run side-by-side from the next cell:

1. **Foreground**: `python -m data.collect_gvgai_ascii` launches one Java MCTS process per `(game, variant, split)`, writing `shard_XXXXX/` directories to the VM's **local SSD** at `LOCAL_DATA_DIR/<split>/<game>/<variant>/`. Jobs are interleaved across games (round-robin), so shards from every game begin appearing early.
2. **Background**: `python -m data.upload_shards` polls `LOCAL_DATA_DIR` every few seconds, detects shards whose Java-side writer finished (marker file `player_y.npy` present + stable), and rsyncs each to the matching path under `DRIVE_DATA_DIR`. After each successful rsync it drops a `.uploaded` sentinel in the local shard dir so re-scans skip it.

The training notebook pulls from Drive, so the trainer sees new shards from every game within `poll_seconds + refresh_every * step_time` of their local finalization.

Both layers are idempotent/resumable:

- Collector: on restart, `ShardAccumulator` picks up from the next free `shard_XXXXX` index.
- Uploader: on restart, any shard without `.uploaded` gets re-rsynced (rsync writes to a temp file and renames on success, so partial transfers never poison Drive).

### Tuning knobs (defined at the top of the next cell)

- `GAMES`: comma-separated games with mappings under `world_model/ascii/mappings/<game>.json`.
- `VARIANTS`: VGDL physics variants per game. `stock` wins fast (few wasted ticks); `physics_a|b|c` usually time out at 2000 ticks per episode because MCTS loses.
- `FRAMES_PER_GAME`: total frames per game, split 90/10 train/test, then split evenly across variants.
- `MCTS_MS`: per-tick MCTS budget in ms. Main bottleneck. `40` = stronger play, `20` = ~2× faster, `10` = ~3× faster with noisier trajectories (still fine for VAE training).
- `CHUNK_SIZE`: frames per shard. Smaller = faster first-shard-on-Drive turnaround for the trainer, more rsync calls. 5000 is a good default; drop to 2000 if you want the trainer to see data sooner.
- `JOBS`: auto-set to `os.cpu_count()` below. Each `(game, variant, split)` is an independent Java process; parallelism scales ~linearly until the job slots or vCPUs are exhausted.
- `UPLOADER_POLL_SECONDS`: how often the background uploader scans for finished shards.

### Smoke-test vs production defaults

The next cell ships with a **smoke-test budget** (`stock` variant only, ~40k frames/game, `MCTS_MS=20`) that finishes in ~15–30 min on a 4-vCPU Colab VM and verifies the full pipeline end-to-end. For a production dataset:

```python
GAMES = 'aliens,chopper,waves'
VARIANTS = 'stock,physics_a,physics_b,physics_c'
FRAMES_PER_GAME = 500_000
MCTS_MS = 40
```

That's ~1.5M frames and takes ~2–8 h depending on vCPU count. Colab sessions time out at ~12 h, so for production prefer to **split collection across sessions by game** (e.g. one session with `GAMES = 'aliens'`, next with `'chopper'`, etc.).

In [ ]:
import os
import subprocess
import sys

GAMES = 'aliens,chopper,waves'
VARIANTS = 'stock'
FRAMES_PER_GAME = 40_000
MCTS_MS = 20
CHUNK_SIZE = 5000
JOBS = os.cpu_count() or 2
UPLOADER_POLL_SECONDS = 10

LOCAL_DATA_DIR.mkdir(parents=True, exist_ok=True)
(LOCAL_DATA_DIR / 'train').mkdir(parents=True, exist_ok=True)
(LOCAL_DATA_DIR / 'test').mkdir(parents=True, exist_ok=True)

print(f'vCPUs detected: {os.cpu_count()}  ->  --jobs={JOBS}')
print(f'plan          : games=[{GAMES}]  variants=[{VARIANTS}]  {FRAMES_PER_GAME:,} frames/game  mcts_ms={MCTS_MS}')
print(f'local out-root: {LOCAL_DATA_DIR}')
print(f'drive mirror  : {DRIVE_DATA_DIR}')

if UPLOADER_STOP_FILE.exists():
    UPLOADER_STOP_FILE.unlink()

uploader_log = open(UPLOADER_LOG, 'w', buffering=1)
uploader = subprocess.Popen(
    [
        sys.executable, '-m', 'data.upload_shards',
        '--local-root', str(LOCAL_DATA_DIR),
        '--remote-root', str(DRIVE_DATA_DIR),
        '--poll-seconds', str(UPLOADER_POLL_SECONDS),
        '--stop-file', str(UPLOADER_STOP_FILE),
        '--verbose',
    ],
    cwd=str(REPO_DIR),
    stdout=uploader_log, stderr=subprocess.STDOUT,
)
print(f'uploader pid  : {uploader.pid}  (log -> {UPLOADER_LOG})')

try:
    rc = subprocess.run(
        [
            sys.executable, '-m', 'data.collect_gvgai_ascii',
            '--games', GAMES,
            '--variants', VARIANTS,
            '--frames-per-game', str(FRAMES_PER_GAME),
            '--mcts-ms', str(MCTS_MS),
            '--chunk-size', str(CHUNK_SIZE),
            '--out-root', str(LOCAL_DATA_DIR),
            '--gvgai-root', str(GVGAI_DIR),
            '--jobs', str(JOBS),
        ],
        cwd=str(REPO_DIR),
    ).returncode
    print(f'\ncollector exit = {rc}')
finally:
    UPLOADER_STOP_FILE.touch()
    try:
        uploader.wait(timeout=max(60, UPLOADER_POLL_SECONDS * 3))
    except subprocess.TimeoutExpired:
        print('uploader did not exit in time; terminating')
        uploader.terminate()
        uploader.wait(timeout=30)
    uploader_log.close()

print('\nfinal flush (sync any stragglers that finished after the uploader stopped)')
subprocess.run(
    [
        sys.executable, '-m', 'data.upload_shards',
        '--local-root', str(LOCAL_DATA_DIR),
        '--remote-root', str(DRIVE_DATA_DIR),
        '--stability-seconds', '0',
        '--once',
        '--verbose',
    ],
    cwd=str(REPO_DIR),
    check=False,
)

## 7. Summarise what was written

In [ ]:
import numpy as np
from collections import Counter

def summarise(label: str, root: Path) -> None:
    if not root.is_dir():
        print(f'{label:<6} {root} -> (missing)')
        return
    per_game = Counter()
    total_frames = 0
    for obs_path in root.rglob('shard_*/obs.npy'):
        game = obs_path.parents[2].name if obs_path.parents[2] != root else obs_path.parent.parent.name
        arr = np.load(obs_path, mmap_mode='r')
        per_game[game] += int(arr.shape[0])
        total_frames += int(arr.shape[0])
    print(f'{label:<6} {root} -> {total_frames:,} frames')
    for g, n in sorted(per_game.items()):
        print(f'  {g:>16}: {n:,}')

summarise('local',  LOCAL_DATA_DIR / 'train')
summarise('local',  LOCAL_DATA_DIR / 'test')
summarise('drive',  DRIVE_DATA_DIR / 'train')
summarise('drive',  DRIVE_DATA_DIR / 'test')